# Extract Data From API

In [0]:
import requests, json
import pyspark.sql.functions as F

url="https://api.coingecko.com/api/v3/coins/markets"

params = {
  "vs_currency": "eur",
  "order": "market_cap_desc",
  "per_page": "100",
  "page": "1",
  "sparkline": "false"
}
# Fetch from API
response = requests.get(url, params=params)
data = response.json()

records = [{"raw_payload": json.dumps(coin)} for coin in data]

df = spark.createDataFrame(records).withColumn(
    "_ingested_at", F.current_timestamp()
).withColumn("ingestion_date", F.to_date(F.current_timestamp())            
).withColumn(
    "_source", F.lit("coingecko_api")
)

#display(df.limit(5))

## Quality checks

In [0]:
df_count = df.count()
null_count = df.filter(F.col("raw_payload").isNull()).count()

assert df_count > 0, "No data received from API"
assert null_count == 0, f"{null_count} null payloads found"

# Store the raw Data to Bronze table

In [0]:

df.write.mode("append").format("delta").partitionBy("ingestion_date").saveAsTable("crypto.bronze.brz_crypto")


In [0]:
%sql

SELECT * FROM crypto.bronze.brz_crypto
LIMIT 10